# Play-by-Play Event Exploration

## NBA Player Impact and Lineup Optimization Dashboard

This notebook inspects the structure and logic of the `play_by_play` table, which serves as the event-level foundation of the project.

The goal is to understand how game events are stored, ordered, and attributed to teams and players, so that substitutions, possessions, and scoring progression can be reconstructed accurately in later notebooks.

## Why this step matters

Lineup analytics depends on correctly reconstructing who was on the floor, when substitutions occurred, and how score and possessions evolved over time.

Before building stint-level datasets, we need to validate that the play-by-play table contains enough information to support:

- chronological event ordering
- game and period segmentation
- player and team attribution
- substitution detection
- score progression
- event classification for later possession logic

This notebook focuses on understanding those mechanics in a structured and reproducible way.

## Objectives

By the end of this notebook, we want to:

- inspect the schema of the `play_by_play` table
- identify the columns needed for event sequencing
- examine how scoring and substitutions are represented
- explore event descriptions and event types
- test a few single-game extracts to verify event flow
- define the fields that will be required for stint construction in Notebook 03

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

DB_PATH = "../data/raw/nba.sqlite"
conn = sqlite3.connect(DB_PATH)

## Inspect the play-by-play schema

We begin by reviewing the column structure of the `play_by_play` table.

The main purpose of this section is to identify the columns that may contain:

- game identifiers
- event sequence
- period and game clock
- player and team IDs
- event descriptions
- score updates

In [2]:
pbp_schema = pd.read_sql("PRAGMA table_info(play_by_play);", conn)
pbp_schema

,cid,name,type,notnull,dflt_value,pk
0,0,game_id,TEXT,0,None,0
1,1,eventnum,INTEGER,0,None,0
2,2,eventmsgtype,INTEGER,0,None,0
3,3,eventmsgactiontype,INTEGER,0,None,0
4,4,period,INTEGER,0,None,0
5,5,wctimestring,TEXT,0,None,0
6,6,pctimestring,TEXT,0,None,0
7,7,homedescription,TEXT,0,None,0
8,8,neutraldescription,TEXT,0,None,0
9,9,visitordescription,TEXT,0,None,0


## Initial schema observations

The `play_by_play` table contains the key fields required for downstream event reconstruction:

- `game_id` for identifying games
- `eventnum` for ordering events
- `eventmsgtype` and `eventmsgactiontype` for classifying actions
- `period` and `pctimestring` for game chronology
- `score` and `scoremargin` for tracking scoring progression
- `homedescription`, `visitordescription`, and `neutraldescription` for event text
- `player1_*`, `player2_*`, and `player3_*` for player and team attribution

This is a strong foundation for constructing possession- and stint-level analytics.

## Preview the raw play-by-play table 

A global preview helps verify how these fields appear in practice before focusing on one game

In [6]:
pbp_preview = pd.read_sql("SELECT * FROM play_by_play LIMIT 10;", conn)
pbp_preview

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,score,scoremargin,person1type,player1_id,player1_name,player1_team_id,player1_team_city,player1_team_nickname,player1_team_abbreviation,person2type,player2_id,player2_name,player2_team_id,player2_team_city,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0029600012,0,12,0,1,14:43 PM,12:00,None,Start of 1st Period (14:43 PM EST),None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
1,0029600012,2,10,0,1,14:50 PM,12:00,Jump Ball O'Neal vs. Kleine: Tip to Cassell,None,None,None,None,4.0,406,Shaquille O'Neal,1610612747.0,Los Angeles,Lakers,LAL,5.0,170,Joe Kleine,1610612756.0,Phoenix,Suns,PHX,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0
2,0029600012,3,2,1,1,14:51 PM,11:45,None,None,MISS Cassell 15' Jump Shot,None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
3,0029600012,4,4,0,1,14:51 PM,11:43,O'Neal REBOUND (Off:0 Def:1),None,None,None,None,4.0,406,Shaquille O'Neal,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
4,0029600012,5,2,1,1,14:51 PM,11:29,MISS Ceballos 26' 3PT Jump Shot,None,None,None,None,4.0,76,Cedric Ceballos,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
5,0029600012,6,4,0,1,14:51 PM,11:27,None,None,Cassell REBOUND (Off:0 Def:1),None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
6,0029600012,7,6,1,1,14:51 PM,11:14,Van Exel P.FOUL (P1.T1),None,None,None,None,4.0,89,Nick Van Exel,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
7,0029600012,8,5,1,1,14:52 PM,11:08,None,None,Cassell Bad Pass Turnover (P1.T1),None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
8,0029600012,9,2,5,1,14:52 PM,10:49,MISS Ceballos 1' Layup,None,Horry BLOCK (1 BLK),None,None,4.0,76,Cedric Ceballos,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,5.0,109,Robert Horry,1610612756.0,Phoenix,Suns,PHX,0
9,0029600012,10,4,0,1,14:52 PM,10:49,LAKERS Rebound,None,None,None,None,2.0,1610612747,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0


## Select one game for detailed inspection 

Event logic is easier to understand at the game level than from a global preview. We first sample a game ID  and then inspect all events from that game in chronological order.

In [7]:
sample_games = pd.read_sql(
    """
    SELECT DISTINCT game_id
    FROM play_by_play
    ORDER BY game_id
    LIMIT 10;
    """,
    conn
)
sample_games

,game_id
0,0011300001
1,0011300002
2,0011300003
3,0011300004
4,0011300005
5,0011300006
6,0011300007
7,0011300008
8,0011300009
9,0011300010


In [5]:
sample_game_id = sample_games.loc[0, "game_id"]

sample_game = pd.read_sql(
    f"""
    SELECT *
    FROM play_by_play
    WHERE game_id = '{sample_game_id}'
    """,
    conn
)

sample_game.head(20)

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,score,scoremargin,person1type,player1_id,player1_name,player1_team_id,player1_team_city,player1_team_nickname,player1_team_abbreviation,person2type,player2_id,player2_name,player2_team_id,player2_team_city,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0029600012,0,12,0,1,14:43 PM,12:00,None,Start of 1st Period (14:43 PM EST),None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
1,0029600012,2,10,0,1,14:50 PM,12:00,Jump Ball O'Neal vs. Kleine: Tip to Cassell,None,None,None,None,4.0,406,Shaquille O'Neal,1610612747.0,Los Angeles,Lakers,LAL,5.0,170,Joe Kleine,1610612756.0,Phoenix,Suns,PHX,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0
2,0029600012,3,2,1,1,14:51 PM,11:45,None,None,MISS Cassell 15' Jump Shot,None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
3,0029600012,4,4,0,1,14:51 PM,11:43,O'Neal REBOUND (Off:0 Def:1),None,None,None,None,4.0,406,Shaquille O'Neal,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
4,0029600012,5,2,1,1,14:51 PM,11:29,MISS Ceballos 26' 3PT Jump Shot,None,None,None,None,4.0,76,Cedric Ceballos,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
5,0029600012,6,4,0,1,14:51 PM,11:27,None,None,Cassell REBOUND (Off:0 Def:1),None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
6,0029600012,7,6,1,1,14:51 PM,11:14,Van Exel P.FOUL (P1.T1),None,None,None,None,4.0,89,Nick Van Exel,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
7,0029600012,8,5,1,1,14:52 PM,11:08,None,None,Cassell Bad Pass Turnover (P1.T1),None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
8,0029600012,9,2,5,1,14:52 PM,10:49,MISS Ceballos 1' Layup,None,Horry BLOCK (1 BLK),None,None,4.0,76,Cedric Ceballos,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,5.0,109,Robert Horry,1610612756.0,Phoenix,Suns,PHX,0
9,0029600012,10,4,0,1,14:52 PM,10:49,LAKERS Rebound,None,None,None,None,2.0,1610612747,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0


In [8]:
sample_game_id=sample_games.loc[0,"game_id"]
sample_game_id

'0011300001'

In [9]:
sample_game=pd.read_sql(
    f"""
    SELECT *
    FROM play_by_play
    WHERE game_id='{sample_game_id}'
    ORDER BY period, eventnum;
    """,
    conn
)

sample_game.head(20)

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,score,scoremargin,person1type,player1_id,player1_name,player1_team_id,player1_team_city,player1_team_nickname,player1_team_abbreviation,person2type,player2_id,player2_name,player2_team_id,player2_team_city,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0011300001,2,12,0,1,1:04 PM,12:00,None,Start of 1st Period (1:04 PM EST),None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
1,0011300001,3,10,0,1,1:05 PM,12:00,Jump Ball Zoric vs. Ibaka: Tip to Jackson,None,None,None,None,4.0,42540,Luka Zoric,12321.0,Istanbul,Fenerbahce Ulker,FBU,5.0,201586,Serge Ibaka,1610612760.0,Oklahoma City,Thunder,OKC,5.0,202704,Reggie Jackson,1610612760.0,Oklahoma City,Thunder,OKC,0
2,0011300001,4,1,1,1,1:05 PM,11:46,None,None,Ibaka 17' Jump Shot (2 PTS) (Jackson 1 AST),2 - 0,-2,5.0,201586,Serge Ibaka,1610612760.0,Oklahoma City,Thunder,OKC,5.0,202704,Reggie Jackson,1610612760.0,Oklahoma City,Thunder,OKC,0.0,0,None,None,None,None,None,0
3,0011300001,5,1,46,1,1:06 PM,11:31,Preldzic 7' Running Jump Shot (2 PTS),None,None,2 - 2,TIE,4.0,42545,Emir Preldzic,12321.0,Istanbul,Fenerbahce Ulker,FBU,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
4,0011300001,6,6,2,1,1:06 PM,11:31,None,None,Durant S.FOUL (P1.T1),None,None,5.0,201142,Kevin Durant,1610612760.0,Oklahoma City,Thunder,OKC,4.0,42545,Emir Preldzic,12321.0,Istanbul,Fenerbahce Ulker,FBU,1.0,0,None,None,None,None,None,0
5,0011300001,7,3,10,1,1:06 PM,11:31,MISS Preldzic Free Throw 1 of 1,None,None,None,None,4.0,42545,Emir Preldzic,12321.0,Istanbul,Fenerbahce Ulker,FBU,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
6,0011300001,9,4,0,1,1:06 PM,11:29,None,None,Jackson REBOUND (Off:0 Def:1),None,None,5.0,202704,Reggie Jackson,1610612760.0,Oklahoma City,Thunder,OKC,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
7,0011300001,10,2,1,1,1:06 PM,11:15,None,None,MISS Durant 13' Jump Shot,None,None,5.0,201142,Kevin Durant,1610612760.0,Oklahoma City,Thunder,OKC,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
8,0011300001,11,4,0,1,1:07 PM,11:14,None,None,Perkins REBOUND (Off:1 Def:0),None,None,5.0,2570,Kendrick Perkins,1610612760.0,Oklahoma City,Thunder,OKC,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
9,0011300001,12,6,1,1,1:07 PM,11:14,Zoric P.FOUL (P1.T1),None,None,None,None,4.0,42540,Luka Zoric,12321.0,Istanbul,Fenerbahce Ulker,FBU,5.0,2570,Kendrick Perkins,1610612760.0,Oklahoma City,Thunder,OKC,1.0,0,None,None,None,None,None,0


##  Build a focused event view

The full raw table contains many fields, but for event interpretation we only need a smaller set of core columns.

This focused view makes it easier to inspect:

    -event sequence 
    -game clock 
    -event descriptions
    -score changes 
    -player references

In [10]:
core_cols = [
    "game_id",
    "eventnum",
    "period",
    "pctimestring",
    "eventmsgtype",
    "eventmsgactiontype",
    "score",
    "scoremargin",
    "homedescription",
    "visitordescription",
    "neutraldescription",
    "player1_id",
    "player1_name",
    "player1_team_id",
    "player2_id",
    "player2_name",
    "player2_team_id",
    "player3_id",
    "player3_name",
    "player3_team_id",
]

sample_game_core = sample_game[core_cols].copy()
sample_game_core.head(30)

,game_id,eventnum,period,pctimestring,eventmsgtype,eventmsgactiontype,score,scoremargin,homedescription,visitordescription,neutraldescription,player1_id,player1_name,player1_team_id,player2_id,player2_name,player2_team_id,player3_id,player3_name,player3_team_id
0,0011300001,2,1,12:00,12,0,None,None,None,None,Start of 1st Period (1:04 PM EST),0,None,None,0,None,None,0,None,None
1,0011300001,3,1,12:00,10,0,None,None,Jump Ball Zoric vs. Ibaka: Tip to Jackson,None,None,42540,Luka Zoric,12321.0,201586,Serge Ibaka,1610612760.0,202704,Reggie Jackson,1610612760.0
2,0011300001,4,1,11:46,1,1,2 - 0,-2,None,Ibaka 17' Jump Shot (2 PTS) (Jackson 1 AST),None,201586,Serge Ibaka,1610612760.0,202704,Reggie Jackson,1610612760.0,0,None,None
3,0011300001,5,1,11:31,1,46,2 - 2,TIE,Preldzic 7' Running Jump Shot (2 PTS),None,None,42545,Emir Preldzic,12321.0,0,None,None,0,None,None
4,0011300001,6,1,11:31,6,2,None,None,None,Durant S.FOUL (P1.T1),None,201142,Kevin Durant,1610612760.0,42545,Emir Preldzic,12321.0,0,None,None
5,0011300001,7,1,11:31,3,10,None,None,MISS Preldzic Free Throw 1 of 1,None,None,42545,Emir Preldzic,12321.0,0,None,None,0,None,None
6,0011300001,9,1,11:29,4,0,None,None,None,Jackson REBOUND (Off:0 Def:1),None,202704,Reggie Jackson,1610612760.0,0,None,None,0,None,None
7,0011300001,10,1,11:15,2,1,None,None,None,MISS Durant 13' Jump Shot,None,201142,Kevin Durant,1610612760.0,0,None,None,0,None,None
8,0011300001,11,1,11:14,4,0,None,None,None,Perkins REBOUND (Off:1 Def:0),None,2570,Kendrick Perkins,1610612760.0,0,None,None,0,None,None
9,0011300001,12,1,11:14,6,1,None,None,Zoric P.FOUL (P1.T1),None,None,42540,Luka Zoric,12321.0,2570,Kendrick Perkins,1610612760.0,0,None,None


## Initial event stream interpretation 

At this stage, we want to visually inspect the first game sample and answer the following questions:

    -Does 'eventnum' appear to order events correctly?
    -Does 'pctimestring' represent time remaining in the period? 
    -Are scoring updates visible in the 'score' field? 
    -Do substituition appear in the description field? 
    -Are players associated with events through 'player1','player2', and 'player3' in a consistent way? 
    
These observations will guide the event-mapping logic used in later notebooks

## Review event type frequency

The next step is to understand the distribution of high-level event type codes. This helps identify the main classes of events recorded in the table and provides a first look at whichcodes may correspond to shots, rebounds, fouls. substitutions, and period makers.

In [11]:
event_type_counts=(
    sample_game["eventmsgtype"]
    .value_counts()
    .sort_index()
    .rename_axis("eventmsgtype")
    .reset_index(name="count")
)

event_type_counts

,eventmsgtype,count
0,1,62
1,2,80
2,3,59
3,4,98
4,5,36
5,6,50
6,7,1
7,8,35
8,9,10
9,10,2


In [12]:
action_type_counts=(
    sample_game["eventmsgtype"]
    .value_counts()
    .sort_index()
    .rename_axis("eventmsgtype")
    .reset_index(name="count")
)

action_type_counts

,eventmsgtype,count
0,1,62
1,2,80
2,3,59
3,4,98
4,5,36
5,6,50
6,7,1
7,8,35
8,9,10
9,10,2


## Why event type codes matter 

The 'eventmsgtype' field will likely become one of the main tools for identifying event categories programatically. 

However, event codes should not be interpreted blindly. They should be cross-validated against the text descriptions and player fields to ensure the mapping is correct. 

For that reason, the next sections inspect event descriptions directly.

## Explore description fields

The play_by_play table includes three text description fields:

    -'homedescription'
    -'visitordescription'
    -'neutraldescription'
    
Together, these should help identify:

    -made and missed shots
    -fouls
    -rebounds
    -substitutions
    -period starts and ends
    -jump balls 
    -timeouts
    
We begin by checking how often each description field is populated.



In [14]:
description_summary=pd.DataFrame({
    "column":["homedescription","visitordescription","neutraldescription"],
    "non_null_count":[
        sample_game["homedescription"].notna().sum(),
        sample_game["visitordescription"].notna().sum(),
        sample_game["neutraldescription"].notna().sum()
    ]
})

description_summary

,column,non_null_count
0,homedescription,218
1,visitordescription,243
2,neutraldescription,10


In [15]:
sample_game_core[
    ["eventnum","period","pctimestring","homedescription","visitordescription", "neutraldescription","score"]
].head(40)

,eventnum,period,pctimestring,homedescription,visitordescription,neutraldescription,score
0,2,1,12:00,None,None,Start of 1st Period (1:04 PM EST),None
1,3,1,12:00,Jump Ball Zoric vs. Ibaka: Tip to Jackson,None,None,None
2,4,1,11:46,None,Ibaka 17' Jump Shot (2 PTS) (Jackson 1 AST),None,2 - 0
3,5,1,11:31,Preldzic 7' Running Jump Shot (2 PTS),None,None,2 - 2
4,6,1,11:31,None,Durant S.FOUL (P1.T1),None,None
5,7,1,11:31,MISS Preldzic Free Throw 1 of 1,None,None,None
6,9,1,11:29,None,Jackson REBOUND (Off:0 Def:1),None,None
7,10,1,11:15,None,MISS Durant 13' Jump Shot,None,None
8,11,1,11:14,None,Perkins REBOUND (Off:1 Def:0),None,None
9,12,1,11:14,Zoric P.FOUL (P1.T1),None,None,None


## Search for substitution event

Substitutions are the most important event class for lineup reconstruction.

In this section, we search the description fields for events containing the text "SUB" to verify:

    -whether substitutions are explicity described.
    -which player fields are populated during substitution rows.
    -wheter the team making the substitution can be identified.

In [18]:
sub_mask=(
    sample_game["homedescription"].astype(str).str.contains("SUB", case=False, na=False) |
    sample_game["visitordescription"].astype(str).str.contains("SUB", case=False, na=False) |
    sample_game["neutraldescription"].astype(str).str.contains("SUB", case=False, na=False) 
 )

sub_events=sample_game.loc[sub_mask, core_cols].copy()
sub_events.head(20)

,game_id,eventnum,period,pctimestring,eventmsgtype,eventmsgactiontype,score,scoremargin,homedescription,visitordescription,neutraldescription,player1_id,player1_name,player1_team_id,player2_id,player2_name,player2_team_id,player3_id,player3_name,player3_team_id
38,0011300001,44,1,7:35,8,0,None,None,SUB: Vidmar FOR Zoric,None,None,42540,Luka Zoric,12321.0,42538,Gasper Vidmar,12321.0,0,None,None
60,0011300001,69,1,4:35,8,0,None,None,None,SUB: Collison FOR Perkins,None,2570,Kendrick Perkins,1610612760.0,2555,Nick Collison,1610612760.0,0,None,None
66,0011300001,77,1,3:57,8,0,None,None,None,SUB: Lamb FOR Durant,None,201142,Kevin Durant,1610612760.0,203087,Jeremy Lamb,1610612760.0,0,None,None
69,0011300001,81,1,3:57,8,0,None,None,SUB: Mahmutoglu FOR Preldzic,None,None,42545,Emir Preldzic,12321.0,42535,Melih Mahmutoglu,12321.0,0,None,None
77,0011300001,92,1,3:09,8,0,None,None,SUB: Sipahi FOR McCalebb,None,None,42531,Bo McCalebb,12321.0,42542,Kenan Sipahi,12321.0,0,None,None
84,0011300001,103,1,2:13,8,0,None,None,None,SUB: Thabeet FOR Ibaka,None,201586,Serge Ibaka,1610612760.0,201934,Hasheem Thabeet,1610612760.0,0,None,None
93,0011300001,113,1,1:28,8,0,None,None,SUB: Turkyilmaz FOR Kleiza,None,None,42536,None,12321.0,42537,Izzet Turkyilmaz,12321.0,0,None,None
107,0011300001,130,1,0:21,8,0,None,None,SUB: Birsen FOR Bogdanovic,None,None,42544,Bojan Bogdanovic,12321.0,42541,James Birsen,12321.0,0,None,None
127,0011300001,158,2,10:32,8,0,None,None,SUB: Zoric FOR Vidmar,None,None,42538,Gasper Vidmar,12321.0,42540,Luka Zoric,12321.0,0,None,None
143,0011300001,177,2,8:20,8,0,None,None,SUB: Preldzic FOR Turkyilmaz,None,None,42537,Izzet Turkyilmaz,12321.0,42545,Emir Preldzic,12321.0,0,None,None


In [19]:
sub_events[[
    "eventnum",
    "period",
    "pctimestring",
    "homedescription",
    "visitordescription",
    "neutraldescription",
    "player1_id",
    "player1_name",
    "player1_team_id",
    "player2_id",
    "player2_name",
    "player2_team_id",
    "player3_id",
    "player3_name",
    "player3_team_id"
]].head(20)

,eventnum,period,pctimestring,homedescription,visitordescription,neutraldescription,player1_id,player1_name,player1_team_id,player2_id,player2_name,player2_team_id,player3_id,player3_name,player3_team_id
38,44,1,7:35,SUB: Vidmar FOR Zoric,None,None,42540,Luka Zoric,12321.0,42538,Gasper Vidmar,12321.0,0,None,None
60,69,1,4:35,None,SUB: Collison FOR Perkins,None,2570,Kendrick Perkins,1610612760.0,2555,Nick Collison,1610612760.0,0,None,None
66,77,1,3:57,None,SUB: Lamb FOR Durant,None,201142,Kevin Durant,1610612760.0,203087,Jeremy Lamb,1610612760.0,0,None,None
69,81,1,3:57,SUB: Mahmutoglu FOR Preldzic,None,None,42545,Emir Preldzic,12321.0,42535,Melih Mahmutoglu,12321.0,0,None,None
77,92,1,3:09,SUB: Sipahi FOR McCalebb,None,None,42531,Bo McCalebb,12321.0,42542,Kenan Sipahi,12321.0,0,None,None
84,103,1,2:13,None,SUB: Thabeet FOR Ibaka,None,201586,Serge Ibaka,1610612760.0,201934,Hasheem Thabeet,1610612760.0,0,None,None
93,113,1,1:28,SUB: Turkyilmaz FOR Kleiza,None,None,42536,None,12321.0,42537,Izzet Turkyilmaz,12321.0,0,None,None
107,130,1,0:21,SUB: Birsen FOR Bogdanovic,None,None,42544,Bojan Bogdanovic,12321.0,42541,James Birsen,12321.0,0,None,None
127,158,2,10:32,SUB: Zoric FOR Vidmar,None,None,42538,Gasper Vidmar,12321.0,42540,Luka Zoric,12321.0,0,None,None
143,177,2,8:20,SUB: Preldzic FOR Turkyilmaz,None,None,42537,Izzet Turkyilmaz,12321.0,42545,Emir Preldzic,12321.0,0,None,None


In [30]:
sub_clean = sub_events.copy()

sub_clean["player_out"] = sub_clean["player1_name"]
sub_clean["player_in"] = sub_clean["player2_name"]
sub_clean["team_id"] = sub_clean["player1_team_id"]

sub_clean[[
    "game_id",
    "eventnum",
    "period",
    "pctimestring",
    "team_id",
    "player_out",
    "player_in"
]].head(10) 
    


,game_id,eventnum,period,pctimestring,team_id,player_out,player_in
38,0011300001,44,1,7:35,12321.0,Luka Zoric,Gasper Vidmar
60,0011300001,69,1,4:35,1610612760.0,Kendrick Perkins,Nick Collison
66,0011300001,77,1,3:57,1610612760.0,Kevin Durant,Jeremy Lamb
69,0011300001,81,1,3:57,12321.0,Emir Preldzic,Melih Mahmutoglu
77,0011300001,92,1,3:09,12321.0,Bo McCalebb,Kenan Sipahi
84,0011300001,103,1,2:13,1610612760.0,Serge Ibaka,Hasheem Thabeet
93,0011300001,113,1,1:28,12321.0,None,Izzet Turkyilmaz
107,0011300001,130,1,0:21,12321.0,Bojan Bogdanovic,James Birsen
127,0011300001,158,2,10:32,12321.0,Gasper Vidmar,Luka Zoric
143,0011300001,177,2,8:20,12321.0,Izzet Turkyilmaz,Emir Preldzic


### Interpretation

Substitution events are clearly represented using `eventmsgtype = 8`, and the player roles are consistently captured through `player1` (out) and `player2` (in).

This confirms that lineup transitions can be reconstructed programmatically without relying on text parsing.


## Substitution rule definition

Based on inspection of substitution events:

- `eventmsgtype = 8` identifies substitution events
- `player1_name` corresponds to the player leaving the game
- `player2_name` corresponds to the player entering the game
- `player1_team_id` identifies the team making the substitution

This allows us to construct structured substitution events without relying on text parsing.

This mapping will be used in Notebook 03 to update the on-court lineup state over time.

## Search for scoring events

To build stint-level point differential later, we need to understand how scoring events appear in the event stream. This section reviews rows with non-nul scores and compares them with event descriptions.

In [21]:
score_events=sample_game.loc[sample_game["score"].notna(),core_cols].copy()
score_events.head(20)

,game_id,eventnum,period,pctimestring,eventmsgtype,eventmsgactiontype,score,scoremargin,homedescription,visitordescription,neutraldescription,player1_id,player1_name,player1_team_id,player2_id,player2_name,player2_team_id,player3_id,player3_name,player3_team_id
2,0011300001,4,1,11:46,1,1,2 - 0,-2,None,Ibaka 17' Jump Shot (2 PTS) (Jackson 1 AST),None,201586,Serge Ibaka,1610612760.0,202704,Reggie Jackson,1610612760.0,0,None,None
3,0011300001,5,1,11:31,1,46,2 - 2,TIE,Preldzic 7' Running Jump Shot (2 PTS),None,None,42545,Emir Preldzic,12321.0,0,None,None,0,None,None
12,0011300001,15,1,10:48,1,47,2 - 4,2,Zoric 10' Turnaround Jump Shot (2 PTS) (Preldzic 1 AST),None,None,42540,Luka Zoric,12321.0,42545,Emir Preldzic,12321.0,0,None,None
17,0011300001,20,1,9:50,1,1,2 - 7,5,Kleiza 24' 3PT Jump Shot (3 PTS) (Zoric 1 AST),None,None,42536,None,12321.0,42540,Luka Zoric,12321.0,0,None,None
23,0011300001,26,1,9:34,3,12,3 - 7,4,None,Perkins Free Throw 2 of 2 (1 PTS),None,2570,Kendrick Perkins,1610612760.0,0,None,None,0,None,None
26,0011300001,30,1,9:10,1,1,6 - 7,1,None,Durant 24' 3PT Jump Shot (3 PTS),None,201142,Kevin Durant,1610612760.0,0,None,None,0,None,None
31,0011300001,36,1,8:11,1,1,6 - 10,4,Bogdanovic 24' 3PT Jump Shot (3 PTS) (McCalebb 1 AST),None,None,42544,Bojan Bogdanovic,12321.0,42531,Bo McCalebb,12321.0,0,None,None
42,0011300001,48,1,7:15,1,1,8 - 10,2,None,Durant 16' Jump Shot (5 PTS) (Perkins 1 AST),None,201142,Kevin Durant,1610612760.0,2570,Kendrick Perkins,1610612760.0,0,None,None
46,0011300001,52,1,6:55,1,72,8 - 12,4,Vidmar 1' Putback Layup (2 PTS),None,None,42538,Gasper Vidmar,12321.0,0,None,None,0,None,None
48,0011300001,54,1,6:32,1,7,8 - 14,6,Kleiza 1' Dunk (5 PTS) (Preldzic 2 AST),None,None,42536,None,12321.0,42545,Emir Preldzic,12321.0,0,None,None


In [22]:
score_events[[
    "eventnum",
    "period",
    "pctimestring",
    "eventmsgtype",
    "eventmsgactiontype",
    "homedescription",
    "visitordescription",
    "neutraldescription",
    "score",
    "scoremargin"
]].head(30)

,eventnum,period,pctimestring,eventmsgtype,eventmsgactiontype,homedescription,visitordescription,neutraldescription,score,scoremargin
2,4,1,11:46,1,1,None,Ibaka 17' Jump Shot (2 PTS) (Jackson 1 AST),None,2 - 0,-2
3,5,1,11:31,1,46,Preldzic 7' Running Jump Shot (2 PTS),None,None,2 - 2,TIE
12,15,1,10:48,1,47,Zoric 10' Turnaround Jump Shot (2 PTS) (Preldzic 1 AST),None,None,2 - 4,2
17,20,1,9:50,1,1,Kleiza 24' 3PT Jump Shot (3 PTS) (Zoric 1 AST),None,None,2 - 7,5
23,26,1,9:34,3,12,None,Perkins Free Throw 2 of 2 (1 PTS),None,3 - 7,4
26,30,1,9:10,1,1,None,Durant 24' 3PT Jump Shot (3 PTS),None,6 - 7,1
31,36,1,8:11,1,1,Bogdanovic 24' 3PT Jump Shot (3 PTS) (McCalebb 1 AST),None,None,6 - 10,4
42,48,1,7:15,1,1,None,Durant 16' Jump Shot (5 PTS) (Perkins 1 AST),None,8 - 10,2
46,52,1,6:55,1,72,Vidmar 1' Putback Layup (2 PTS),None,None,8 - 12,4
48,54,1,6:32,1,7,Kleiza 1' Dunk (5 PTS) (Preldzic 2 AST),None,None,8 - 14,6


## Score progression interpretation

The `score` field appears to provide direct score updates during the game.

This is useful because it means point differential can be tracked without fully reconstructing every made shot from scratch.

In later notebooks, this field can be used to compute score change across stints by comparing the score at the start and end of each lineup segment.

## Inspect player roles across events

Because the table includes `player1`, `player2`, and `player3`, it is important to understand how these fields behave for different event categories.

For example:
- a made shot may involve a shooter and an assister
- a foul may involve the fouling player and the fouled player
- a substitution may involve an entering and exiting player

This section provides examples for manual inspection.

In [23]:
sample_game_core[
    [
        "eventnum",
        "period",
        "pctimestring",
        "eventmsgtype",
        "eventmsgactiontype",
        "homedescription",
        "visitordescription",
        "neutraldescription",
        "player1_name",
        "player2_name",
        "player3_name",
        "player1_team_id",
        "player2_team_id",
        "player3_team_id",
        "score"
    ]
].head(50)

,eventnum,period,pctimestring,eventmsgtype,eventmsgactiontype,homedescription,visitordescription,neutraldescription,player1_name,player2_name,player3_name,player1_team_id,player2_team_id,player3_team_id,score
0,2,1,12:00,12,0,None,None,Start of 1st Period (1:04 PM EST),None,None,None,None,None,None,None
1,3,1,12:00,10,0,Jump Ball Zoric vs. Ibaka: Tip to Jackson,None,None,Luka Zoric,Serge Ibaka,Reggie Jackson,12321.0,1610612760.0,1610612760.0,None
2,4,1,11:46,1,1,None,Ibaka 17' Jump Shot (2 PTS) (Jackson 1 AST),None,Serge Ibaka,Reggie Jackson,None,1610612760.0,1610612760.0,None,2 - 0
3,5,1,11:31,1,46,Preldzic 7' Running Jump Shot (2 PTS),None,None,Emir Preldzic,None,None,12321.0,None,None,2 - 2
4,6,1,11:31,6,2,None,Durant S.FOUL (P1.T1),None,Kevin Durant,Emir Preldzic,None,1610612760.0,12321.0,None,None
5,7,1,11:31,3,10,MISS Preldzic Free Throw 1 of 1,None,None,Emir Preldzic,None,None,12321.0,None,None,None
6,9,1,11:29,4,0,None,Jackson REBOUND (Off:0 Def:1),None,Reggie Jackson,None,None,1610612760.0,None,None,None
7,10,1,11:15,2,1,None,MISS Durant 13' Jump Shot,None,Kevin Durant,None,None,1610612760.0,None,None,None
8,11,1,11:14,4,0,None,Perkins REBOUND (Off:1 Def:0),None,Kendrick Perkins,None,None,1610612760.0,None,None,None
9,12,1,11:14,6,1,Zoric P.FOUL (P1.T1),None,None,Luka Zoric,Kendrick Perkins,None,12321.0,1610612760.0,None,None


## Create a preliminary event mapping table

At this stage, we can begin documenting a preliminary mapping between event types and basketball actions.

This mapping does not need to be perfect yet. The goal is to create a first reference that will later support event cleaning and possession logic.

In [31]:
event_mapping = pd.DataFrame({
    "eventmsgtype": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13],
    "event_category": [
        "Made Shot",
        "Missed Shot",
        "Free Throw",
        "Rebound",
        "Turnover",
        "Foul",
        "Violation",
        "Substitution",
        "Timeout",
        "Jump Ball",
        "Start Period",
        "End Period"
    ],
    "used_in_pipeline": [
        "Scoring and possessions",
        "Possessions",
        "Scoring and possessions",
        "Possessions",
        "Possessions",
        "Context and foul logic",
        "Context",
        "Structured substitution extraction for lineup reconstruction",
        "Context",
        "Game state initialization",
        "Period segmentation",
        "Period segmentation"
    ]
})

event_mapping

,eventmsgtype,event_category,used_in_pipeline
0,1,Made Shot,Scoring and possessions
1,2,Missed Shot,Possessions
2,3,Free Throw,Scoring and possessions
3,4,Rebound,Possessions
4,5,Turnover,Possessions
5,6,Foul,Context and foul logic
6,7,Violation,Context
7,8,Substitution,Structured substitution extraction for lineup reconstruction
8,9,Timeout,Context
9,10,Jump Ball,Game state initialization


## Define fields required for stint construction

The main deliverable of this notebook is a validated set of fields that will be used in Notebook 03.

For stint construction, the project needs:
- `game_id`
- `eventnum`
- `period`
- `pctimestring`
- `eventmsgtype`
- event descriptions
- score fields
- player/team reference fields

In [26]:
required_fields_check = pd.DataFrame({
    "required_field_role": [
        "Game identifier",
        "Event order",
        "Period",
        "Period clock",
        "Main event type",
        "Detailed action type",
        "Home description",
        "Visitor description",
        "Neutral description",
        "Score",
        "Score margin",
        "Primary player id",
        "Secondary player id",
        "Tertiary player id",
        "Primary team id"
    ],
    "candidate_column": [
        "game_id",
        "eventnum",
        "period",
        "pctimestring",
        "eventmsgtype",
        "eventmsgactiontype",
        "homedescription",
        "visitordescription",
        "neutraldescription",
        "score",
        "scoremargin",
        "player1_id",
        "player2_id",
        "player3_id",
        "player1_team_id"
    ],
    "exists_in_table": [
        "game_id" in sample_game.columns,
        "eventnum" in sample_game.columns,
        "period" in sample_game.columns,
        "pctimestring" in sample_game.columns,
        "eventmsgtype" in sample_game.columns,
        "eventmsgactiontype" in sample_game.columns,
        "homedescription" in sample_game.columns,
        "visitordescription" in sample_game.columns,
        "neutraldescription" in sample_game.columns,
        "score" in sample_game.columns,
        "scoremargin" in sample_game.columns,
        "player1_id" in sample_game.columns,
        "player2_id" in sample_game.columns,
        "player3_id" in sample_game.columns,
        "player1_team_id" in sample_game.columns
    ]
})

required_fields_check

,required_field_role,candidate_column,exists_in_table
0,Game identifier,game_id,True
1,Event order,eventnum,True
2,Period,period,True
3,Period clock,pctimestring,True
4,Main event type,eventmsgtype,True
5,Detailed action type,eventmsgactiontype,True
6,Home description,homedescription,True
7,Visitor description,visitordescription,True
8,Neutral description,neutraldescription,True
9,Score,score,True


In [32]:
sub_clean.to_csv("../data/interim/substitutions_sample.csv", index=False)

## Key findings

The `play_by_play` table contains the core fields needed to support event-level reconstruction for lineup analytics.

Most importantly, the dataset includes:
- an explicit game identifier
- an event ordering field
- period and game clock information
- score updates
- text descriptions for multiple event perspectives
- player and team references for up to three players per event

This confirms that the database is structurally suitable for reconstructing substitutions, tracking score progression, and building stint-level datasets.

## Notebook takeaway

The play-by-play dataset contains all required components to reconstruct game flow and lineup changes.

Key validated elements:
- Event ordering via `eventnum`
- Score tracking via `score`
- Substitutions via `eventmsgtype = 8`
- Player roles via structured columns

The dataset is suitable for building a stint-level lineup reconstruction pipeline.

## Next step

Notebook 03 will move from exploration to engineering.

The main objective will be to reconstruct game stints by:
- ordering events within a game
- identifying substitutions
- updating the on-court player state
- segmenting games into lineup stints
- calculating score change across those stints